# Node2vec

> [!WARNING]
> This tutorial is experimental and requires `mlx_cluster>=0.1.2` to be installed
> (which requires `mlx>=0.32`).

**Goal:** This tutorial will guide you through implementing node2vec to generate vector embeddings for nodes in a simple undirected graph.

**Concepts:** `MLX`, `Node2vec`

In [1]:
from collections import defaultdict
import mlx.core as mx
from mlx_graphs.datasets import PlanetoidDataset

# Fix the seed so random walks (and results) are reproducible across runs
mx.random.seed(42)

## Dataset

For this first tutorial, we will use the [PlanetoidDataset](https://github.com/kimiyoung/planetoid) collection, which comprises of citation networks for `Cora`, `Pubmed` and `CiteSeer`.

We will be using `Cora` dataset consisting of **2708** nodes and **10,556** edges. The dataset can be easily accessed via `PlanetoidDataset` class

In [2]:
dataset = PlanetoidDataset("Cora")

Loading cora data ... Done


In [3]:
dataset

cora(num_graphs=1)

We can access dataset properties directly from `dataset`object

In [4]:
# Some useful properties
print("Dataset attributes")
print("-" * 20)
print(f"Number of graphs: {len(dataset)}")
print(f"Number of node features: {dataset.num_node_features}")
print(f"Number of edge features: {dataset.num_edge_features}")
print(f"Number of graph features: {dataset.num_graph_features}")
print(f"Number of graph classes to predict: {dataset.num_graph_classes}\n")

# Statistics of the dataset
stats = defaultdict(list)
for g in dataset:
    stats["Mean node degree"].append(g.num_edges / g.num_nodes)
    stats["Mean num of nodes"].append(g.num_nodes)
    stats["Mean num of edges"].append(g.num_edges)

print("Dataset stats")
print("-" * 20)
for k, v in stats.items():
    mean = mx.mean(mx.array(v)).item()
    print(f"{k}: {mean:.2f}")

Dataset attributes
--------------------
Number of graphs: 1
Number of node features: 1433
Number of edge features: 0
Number of graph features: 0
Number of graph classes to predict: 0

Dataset stats
--------------------
Mean node degree: 3.90
Mean num of nodes: 2708.00
Mean num of edges: 10556.00


In [5]:
dataset[0]

GraphData(
	edge_index(shape=(2, 10556), int32)
	node_features(shape=(2708, 1433), float32)
	node_labels(shape=(2708,), int32)
	train_mask(shape=(2708,), bool)
	val_mask(shape=(2708,), bool)
	test_mask(shape=(2708,), bool))

## Creating a simple neural network using node2vec

In [6]:
from mlx_graphs.algorithms import Node2Vec

## Specify hyperparameters for node2vec.

The most important hyperparameters for node2vec are `p` and `q`, which bias each step of the walk based on the node visited just before:
1. `p` (return parameter): controls the likelihood of immediately revisiting the previous node. When `p` is low, the walk is more likely to step back.
2. `q` (in-out parameter): controls whether the walk stays close to the source or moves away from it. When `q` is low, the walk is more likely to move outward (DFS-like exploration); when `q` is high, it stays within the local neighbourhood (BFS-like).
3. `embedding_dim`: dimensions of the embedding vectors
4. `walk_length`: number of nodes to consider in a walk
5. `context_size`: the actual context size which is considered for positive samples. This parameter increases the effective sampling rate by reusing samples across different source nodes.
6. `walks_per_node`: number of walks to start from each node

With `p = q = 1` the walks are uniform random walks (DeepWalk); any other value uses node2vec's biased walks.

In [7]:
embedding_dim = 128
walk_length = 50
context_size = 10
walks_per_node = 10
num_negative_samples = 1
p = 1.0
q = 1.0

## Training loop

Each epoch iterates over all nodes in mini-batches. For every batch, `model.dataloader` samples positive random walks and negative (random) walks, and the model is trained with the negative-sampling loss.

## Benchmark on CPU and GPU

`use_gpu` only selects the device for random-walk sampling. To compare full training on each device, we also set MLX's default device, so embedding lookups, the loss, gradients and optimizer updates run on the same device as the walks.

In [8]:
import mlx.nn as nn
import mlx.optimizers as optim
import time

In [9]:
def training_loop(model, optimizer, num_epochs=10, batch_size=32):
    """Train the model and print the average loss per batch for each epoch."""
    loss_and_grad = nn.value_and_grad(model, model.loss)
    for epoch in range(num_epochs):
        total_loss = 0.0
        num_batches = 0
        for pos, neg in model.dataloader(batch_size=batch_size):
            loss, grad = loss_and_grad(pos, neg)
            optimizer.update(model, grad)
            total_loss += loss.item()
            num_batches += 1
        print(f"Epoch {epoch}: avg loss = {total_loss / num_batches:.5f}")

In [10]:
def benchmark_training(model_creation_fn, optimizer_creation_fn, device="gpu", num_epochs=10, batch_size=32):
    """
    Benchmark full training on the specified device.

    Args:
        model_creation_fn: Function that returns a new model instance
        optimizer_creation_fn: Function that takes model and returns optimizer
        device: 'gpu' or 'cpu'
    """
    print(f"🚀 {device.upper()} Benchmark")
    print("-" * 40)

    use_gpu = device == "gpu"
    previous_device = mx.default_device()
    # Run the whole training step (not just the walks) on the chosen device
    mx.set_default_device(mx.gpu if use_gpu else mx.cpu)
    try:
        # Create fresh model and optimizer for this benchmark
        model = model_creation_fn(use_gpu=use_gpu)
        optimizer = optimizer_creation_fn(model)

        start_time = time.perf_counter()
        training_loop(model, optimizer, num_epochs, batch_size)
        total_time = time.perf_counter() - start_time
    finally:
        mx.set_default_device(previous_device)

    print(f"🎯 {device.upper()} Total Time: {total_time:.3f} seconds")
    print(f"📈 Average per epoch: {total_time / num_epochs:.3f} seconds")

    return total_time

In [11]:
def create_model(use_gpu=False):
    """Create a Node2Vec model with the hyperparameters above."""
    model = Node2Vec(
        edge_index=dataset[0].edge_index,
        num_nodes=dataset[0].num_nodes,
        embedding_dim=embedding_dim,
        walk_length=walk_length,
        context_size=context_size,
        walks_per_node=walks_per_node,
        num_negative_samples=num_negative_samples,
        p=p,
        q=q,
        use_gpu=use_gpu,
    )
    return model

In [12]:
def create_optimizer(model):
    """Create the optimizer used for training."""
    return optim.Adam(learning_rate=0.001)

In [13]:
print("Starting benchmarks...\n")

# GPU benchmark
gpu_time = benchmark_training(create_model, create_optimizer, num_epochs=100, device="gpu")

print("\n" + "=" * 50 + "\n")

# CPU benchmark
cpu_time = benchmark_training(create_model, create_optimizer, num_epochs=100, device="cpu")

# Summary
print("\n📊 BENCHMARK SUMMARY")
print("=" * 50)
print(f"GPU Time:     {gpu_time:.3f}s")
print(f"CPU Time:     {cpu_time:.3f}s")
print(f"GPU Speedup:  {cpu_time / gpu_time:.2f}x faster")

Starting benchmarks...

🚀 GPU Benchmark
----------------------------------------
Epoch 0: avg loss = 1.01090
Epoch 1: avg loss = 0.84746
Epoch 2: avg loss = 0.83582
Epoch 3: avg loss = 0.83097
Epoch 4: avg loss = 0.82852
Epoch 5: avg loss = 0.82687
Epoch 6: avg loss = 0.82551
Epoch 7: avg loss = 0.82421
Epoch 8: avg loss = 0.82339
Epoch 9: avg loss = 0.82295
Epoch 10: avg loss = 0.82249
Epoch 11: avg loss = 0.82216
Epoch 12: avg loss = 0.82136
Epoch 13: avg loss = 0.82147
Epoch 14: avg loss = 0.82176
Epoch 15: avg loss = 0.82079
Epoch 16: avg loss = 0.82055
Epoch 17: avg loss = 0.82026
Epoch 18: avg loss = 0.82018
Epoch 19: avg loss = 0.81956
Epoch 20: avg loss = 0.82008
Epoch 21: avg loss = 0.81998
Epoch 22: avg loss = 0.81891
Epoch 23: avg loss = 0.81926
Epoch 24: avg loss = 0.81937
Epoch 25: avg loss = 0.81913
Epoch 26: avg loss = 0.81912
Epoch 27: avg loss = 0.81905
Epoch 28: avg loss = 0.81851
Epoch 29: avg loss = 0.81885
Epoch 30: avg loss = 0.81858
Epoch 31: avg loss = 0.81833
E